For tutorials on netket, check 
https://netket.readthedocs.io/en/latest/tutorials/index.html

# 0. Check GPU info

In [1]:

import subprocess

# This helper prints the GPU model reported by nvidia-smi.
def get_mdl():
    line_as_bytes = subprocess.check_output("nvidia-smi -L", shell=True)
    line = line_as_bytes.decode("ascii")
    _, line = line.split(":", 1)
    line, _ = line.split("(")
    return line.strip()

def check_memory():
    # Call nvidia-smi to get the total, used, and free GPU memory.
    result = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,memory.used,memory.free', '--format=csv'], stdout=subprocess.PIPE)
    print(result.stdout.decode('utf-8'))
    
check_memory()
print(subprocess.check_output("nvidia-smi -L", shell=True))

memory.total [MiB], memory.used [MiB], memory.free [MiB]
32760 MiB, 24510 MiB, 7730 MiB

b'GPU 0: NVIDIA RTX 5000 Ada Generation (UUID: GPU-3b429101-b7d1-6a66-d897-d93c375836af)\n'


# 1. Lattice setup

In [2]:
import os
import json
import flax
import numpy as np

# Enable NetKet's experimental sharding support before importing JAX/NetKet.
# This is useful when running the same notebook on one or more GPUs.
os.environ['NETKET_EXPERIMENTAL_SHARDING'] = '1'

import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx


print("Sharding is enabled:", nk.config.netket_experimental_sharding)
print("The available GPUs are:", jax.devices())

# Global random key used for reproducible initialization and sampling choices.
key = jax.random.PRNGKey(17)


# Define the square lattice geometry. Change Lx/Ly to study a different system size.
Lx = 4  # Number of sites along x
Ly = 8  # Number of sites along y
basis_vectors = np.array([[1, 0], [0, 1]]) # Standard square lattice primitive vectors
extent = (Lx, Ly) # Lattice extent
graph = nk.graph.Lattice(basis_vectors, extent, pbc=True)
lattice_shape = [Lx,Ly,1]


lattice_row = lattice_shape[0]
lattice_col = lattice_shape[1]
lattice_hgt = lattice_shape[2]

num_sites = graph.n_nodes
# Spin-up and spin-down fermions are represented on two copies of the lattice.
# The disjoint union graph is used by the fermion-hop sampler below.
exchange_graph = nk.graph.disjoint_union(graph, graph)
print("Exchange graph size:", exchange_graph.n_nodes)



∣NK⟩ Tip: You can use flax.linen, flax.nnx and equinox to define neural networks.

Sharding is enabled: True
The available GPUs are: [CudaDevice(id=0)]
Exchange graph size: 64


# 2. Hamiltonian setup

In [3]:
import netket as nk
import netket.experimental

# Electron filling. For this example, n=0.875 corresponds to 1/8 hole doping.
n = 0.875 # hole doping = 1-n
num_electrons = int(n*num_sites) # total number of electrons
num_up = num_electrons//2 # number of spin up electrons
num_dn = num_electrons//2 # number of spin dn electrons

# Hubbard model parameters: nearest-neighbor hopping t and onsite interaction U.
U = 8
t = 1

# Build the fixed-particle-number Hilbert space, split evenly between spins.
hi = nk.hilbert.SpinOrbitalFermions(n_orbitals=graph.n_nodes, s=1/2, n_fermions_per_spin=(num_up, num_dn))
# Construct the Fermi-Hubbard Hamiltonian and convert it to the JAX operator backend.
Hamiltonian = netket.experimental.operator.FermiHubbardJax(hi, t=t, U=U, graph=graph)
Hamiltonian = Hamiltonian.to_jax_operator()

# 3. Create neural network

In [4]:
from Network_lib import *


# Neural-network hyperparameters for the variational wavefunction ansatz.
num_layers = 3 # number of encoder layers
embedding_dim = 32 # embedding dimension
num_kernels = 8 # number of static kernels
patch_dim = 2 # size of a patch = (patch_dim, patch_dim)

num_det_M = 8 # number of delta_M matrices
num_det_F = 8 # number of delta_F matrices
rank = 2 # rank of the delta matrices

# Initialize the Net module with both architecture choices and lattice metadata.
model = Net(num_layers = num_layers,
            embedding_dim = embedding_dim,
            num_kernels = num_kernels,
            patch_dim = patch_dim,
            num_sites = num_sites,
            num_electrons = num_electrons,
            num_det_M = num_det_M,
            num_det_F = num_det_F,
            rank = rank,
            lattice_row = lattice_row,
            lattice_col = lattice_col)


# Use a small batch of random Hilbert states to print a parameter/shape summary.
tabulate_fn = nn.tabulate(model, jax.random.PRNGKey(0))
dummy_input = hi.random_state(jax.random.PRNGKey(0), 7, dtype=jnp.float64)
net_summary = tabulate_fn(dummy_input)
print(net_summary)


                                  Net Summary                                   
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ path          ┃ module        ┃ inputs        ┃ outputs      ┃ params        ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│               │ Net           │ float64[7,64] │ complex128[… │               │
├───────────────┼───────────────┼───────────────┼──────────────┼───────────────┤
│ EmbeddingMod… │ EmbeddingMod… │ float64[7,64] │ float64[7,8… │               │
├───────────────┼───────────────┼───────────────┼──────────────┼───────────────┤
│ EmbeddingMod… │ Dense         │ float64[7,8,… │ float64[7,8… │ bias:         │
│               │               │               │              │ float64[32]   │
│               │               │               │              │ kernel:       │
│               │               │               │              │ float64[8,32] │
│               │          

In [ ]:
# Optional quick smoke test: uncomment this cell to initialize parameters and time one forward pass of the variational wavefunction.

# key, subkey = jax.random.split(key)
# params = model.init(subkey, dummy_input)

# import time
# t0 = time.perf_counter()
# test_wf = model.apply(params,dummy_input)
# t1 = time.perf_counter()
# print('time:',t1-t0)

# print(test_wf)

time: 0.12350470793899149
[-24.94033463+5.36746530e-18j -25.78066123+3.14159265e+00j
 -26.52648509+3.14159265e+00j -23.9853183 +3.54839373e-17j
 -27.09292136+1.26495877e-17j -24.32441722+1.89679831e-17j
 -31.46158066+3.14159265e+00j]


# 4. Variational Monte Carlo setup

In [6]:
# Monte Carlo sampling settings for estimating the variational energy.
n_samples = 4096 # total number of samples 
n_chains = 16 # number of independent Markov chains
chunk_size = 2048 
seed = 42 # random seed, can be other values

# MetropolisFermionHop proposes particle hops along the exchange graph.
sampler = nk.sampler.MetropolisFermionHop(hi, graph=exchange_graph, sweep_size=num_sites, n_chains=n_chains, d_max=1)
# MCState connects the sampler and neural network; n_discard_per_chain warms up each chain.
vstate = nk.vqs.MCState(sampler, model, n_samples=n_samples, n_discard_per_chain=num_sites, chunk_size=chunk_size, seed=seed)

# 5. Job setup

In [7]:
# Job metadata controls output names and the text summary saved with the run.
job_id = 'jupyter_hubbard_test'
job_summary = '4x8 lattice, 1/8 doping, U=8'

# Set continue_training=True to resume from the saved .mpack/.log files in this folder.
old_step=0
continue_training = False


print(job_summary)

# Training artifacts are written under Training_results/<job_id>/.
folder_path = os.path.join(os.getcwd()+'/Training_results/', job_id)
os.makedirs(folder_path, exist_ok=True)

with open(folder_path+"/"+job_id+'_job_summary.txt', "w") as text_file:
    text_file.write(job_summary)

# JsonLog records observables and periodically saves variational parameters.
logger=nk.logging.JsonLog(folder_path+'/'+job_id, mode='write', save_params_every=5, write_every=5, save_params=True)


if continue_training == True:
    # The mpack file stores only the parameters of the variational state.
    with open(folder_path+'/'+job_id+'.mpack', 'rb') as file:
        vstate.variables = flax.serialization.from_bytes(vstate.variables, file.read())
        
    # Load the HistoryDict containing previous training results.
    saved_data = nk.utils.history.HistoryDict.from_file(folder_path+'/'+job_id+'.log')

    energy_dict = saved_data['Energy'].to_dict()
    # energy_dict['Mean'] = np.array(energy_dict['Mean'][0]['real']) + 1j*np.array(energy_dict['Mean'][0]['imag'])  
    energy_dict['Mean'] = np.array(energy_dict['Mean'])
    saved_data['Energy'] = nk.utils.history.History(values=energy_dict)
    logger._data = saved_data

    # Continue the progress counter from the last saved energy measurement.
    old_step = len(saved_data['Energy'].iters)
    logger._old_step = old_step



4x8 lattice, 1/8 doping, U=8


# 6. Optimizer and Runner setup

In [ ]:

import optax
from Optimizer_lib import *

# Base optimizer used inside the driver.
sgd = nk.optimizer.Sgd(learning_rate=1)

# SPRING hyperparameters. Tune these when changing model size or sampling budget.
diag_shift = 1e-3 # lambda regularizer in the inverse
momentum = 0.8 # mu term in the SPRING optimizer
eta = 0.01 # learning rate eta in the SPRING optimizer

# Piecewise schedule for the clipping/regularization coefficient C.
C_schedule = optax.join_schedules(
    schedules=[
        optax.constant_schedule(0.01),  # First phase
        optax.constant_schedule(0.01/9)  # Second phase
    ],
    boundaries=[50_000]  # Switch after 50,000 steps
) # constant C in the SPRING optimizer

# Backward-pass chunking trades memory usage against runtime.
chunk_size_bwd = 2048 

# Driver that performs VMC optimization with stochastic reconfiguration and norm clipping.
gs = VMC_SR_norm_clip(
    hamiltonian=Hamiltonian, 
    optimizer=sgd, 
    diag_shift=diag_shift, 
    momentum=momentum, 
    variational_state=vstate, 
    mode="real",
    eta=eta,
    C = C_schedule,
    chunk_size_bwd=chunk_size_bwd)


Automatic SR implementation choice:  NTK


In [10]:
# Number of optimization steps to run. Lower this for a quick smoke test.
n_iter = 10_000 # number of training iterations

# Keep the logger/driver step numbering consistent when resuming a previous job.
gs._step_count = old_step
gs.run(n_iter=n_iter, out=logger)


  0%|          | 0/10000 [00:00<?, ?it/s]

W0515 13:09:01.023105 3522366 gemm_fusion_autotuner.cc:1163] Compiling 125 configs for 12 fusions on a single thread.


KeyboardInterrupt: 

# 7. Load trained results

In [ ]:
def load_trained_results(job_id,n=100):
    # Read the NetKet JsonLog created during training for the selected job.
    folder_path = os.path.join(os.getcwd()+'/Training_results/', job_id)
    saved_data = nk.utils.history.HistoryDict.from_file(folder_path+'/'+job_id+'.log')
    
    # Extract the energy mean and variance histories from the saved log.
    # e = jnp.array(saved_data['Energy']['Mean'][0]['real'])
    e = jnp.array(saved_data['Energy']['Mean'].real)
    var = jnp.array(saved_data['Energy']['Variance'])
    
    # Smooth the curves with an n-step moving average for easier visualization.
    e_avg = jnp.convolve(e, jnp.ones(n)/n, mode='valid')
    var_avg = jnp.convolve(var, jnp.ones(n)/n, mode='valid')
    return e, e_avg, var, var_avg

In [ ]:
# Load the results for the job configured above.
job_id = 'jupyter_hubbard_test'
e, e_avg, var, var_avg = load_trained_results(job_id,n=100)